# 06 — CatBoost: optimized HRI regression

**Purpose:** Tune CatBoost on the frozen `train.csv` and report grouped cross-validation and frozen `validation.csv` performance. Each year of a `cell_id` stays in the same CV fold. All training and evaluation code is in this notebook.

**Research limit:** `hri_score_0_100` is a constructed same-year index. Better scores show a better approximation of that index, not a forecast of future habitat degradation. The nine inputs match the original notebooks. Keep the test cell locked until one model is selected across all ten notebooks.

## 1 — Imports, paths, target, and predictors

In [ ]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, ParameterGrid, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor

# Find the component root from any working directory inside the repository.
PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p / 'data/splits/hri/model_v1/train.csv').is_file()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p / 'components/habitat_risk' for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                         if (p / 'components/habitat_risk/data/splits/hri/model_v1/train.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run Jupyter from within the repository or component directory.')
DATA_DIR = PROJECT_ROOT / 'data/splits/hri/model_v1'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts/hri/model_training_optimized/catboost'
TARGET = 'hri_score_0_100'
FEATURES = [
    'climate__annual_mean_temperature_c',
    'climate__annual_precipitation_mm',
    'climate__temperature_c_anomaly',
    'climate__temperature_c_trailing5_slope_per_year',
    'climate__precipitation_mm_anomaly',
    'climate__precipitation_mm_trailing5_slope_per_year',
    'terrain__mean_elevation_m',
    'terrain__elevation_sd_m',
    'terrain__mean_slope_degrees',
]
RANDOM_STATE = 42


## 2 — Load and audit the frozen training and validation splits

In [ ]:
def load_split(name):
    frame = pd.read_csv(DATA_DIR / f'{name}.csv')
    required = set(FEATURES + [TARGET, 'cell_id', 'reference_year', 'dataset_split'])
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f'{name} missing columns: {sorted(missing)}')
    if not frame['dataset_split'].eq(name).all():
        raise ValueError(f'{name}: incorrect dataset_split labels')
    if frame[['cell_id', 'reference_year']].duplicated().any():
        raise ValueError(f'{name}: duplicate cell-year keys')
    for col in FEATURES + [TARGET]:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
    frame[FEATURES] = frame[FEATURES].replace([np.inf, -np.inf], np.nan)
    valid = np.isfinite(frame[TARGET].to_numpy(dtype=float))
    if not valid.all():
        print(f'{name}: dropping {(~valid).sum()} rows with invalid target')
        frame = frame.loc[valid].copy()
    if frame.empty:
        raise ValueError(f'{name}: no usable rows')
    return frame

train = load_split('train')
validation = load_split('validation')
if set(train.cell_id) & set(validation.cell_id):
    raise ValueError('A cell_id occurs in both training and validation')
for label, frame in [('train', train), ('validation', validation)]:
    print(label, frame.shape, 'cells:', frame.cell_id.nunique(),
          'missing feature values:', int(frame[FEATURES].isna().sum().sum()))
display(train[FEATURES + [TARGET]].describe().T)


## 3 — Define the preprocessing pipeline and parameter search

In [ ]:
# Imputation is fitted inside each CV training fold. Trees need no scaling.
preprocess = SimpleImputer(strategy='median')

regressor = CatBoostRegressor(loss_function='RMSE', verbose=False, allow_writing_files=False,
                              thread_count=4, random_seed=42)
param_space = {
    'model__iterations': [250, 500, 750],
    'model__depth': [3, 5, 7],
    'model__learning_rate': [0.02, 0.05, 0.1],
    'model__l2_leaf_reg': [3, 10, 30],
}

pipeline = Pipeline([('preprocess', preprocess), ('model', regressor)])

## 4 — Grouped cross-validation and hyperparameter optimization

Search uses only `train.csv`. Mean CV MAE chooses the parameters; each fold fits its own preprocessing. Increase `N_ITER` for a wider search, consistently across models.

In [ ]:
N_FOLDS = 3
N_ITER = 8
if N_FOLDS < 2 or N_FOLDS > train.cell_id.nunique():
    raise ValueError('N_FOLDS must be between 2 and the number of training cells')
cv = GroupKFold(n_splits=N_FOLDS)
search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_space,
    n_iter=min(N_ITER, len(ParameterGrid(param_space))),
    scoring={'MAE': 'neg_mean_absolute_error',
             'RMSE': 'neg_root_mean_squared_error', 'R2': 'r2'},
    refit='MAE', cv=cv, n_jobs=1, random_state=RANDOM_STATE,
    return_train_score=True, error_score='raise', verbose=1,
)
start = time.perf_counter()
search.fit(train[FEATURES], train[TARGET], groups=train.cell_id)
train_seconds = time.perf_counter() - start
model = search.best_estimator_  # Refitted on all training rows only.
print('Best CV MAE:', -search.best_score_)
print('Best parameters:', search.best_params_)
print('Tuning seconds:', round(train_seconds, 2))


## 5 — Training and held-out validation diagnostics

In [ ]:
def score(y, prediction):
    return {'MAE': float(mean_absolute_error(y, prediction)),
            'RMSE': float(np.sqrt(mean_squared_error(y, prediction))),
            'R2': float(r2_score(y, prediction))}

train_pred = model.predict(train[FEATURES])
val_pred = model.predict(validation[FEATURES])
train_metrics = score(train[TARGET], train_pred)
val_metrics = score(validation[TARGET], val_pred)
baseline = DummyRegressor(strategy='mean').fit(train[FEATURES], train[TARGET])
baseline_metrics = score(validation[TARGET], baseline.predict(validation[FEATURES]))
best = search.best_index_
summary = {
    'model': 'catboost', 'n_train': len(train), 'n_validation': len(validation),
    'folds': N_FOLDS, 'candidates': len(search.cv_results_['params']),
    'cv_MAE': float(-search.cv_results_['mean_test_MAE'][best]),
    'cv_MAE_std': float(search.cv_results_['std_test_MAE'][best]),
    'cv_RMSE': float(-search.cv_results_['mean_test_RMSE'][best]),
    'cv_R2': float(search.cv_results_['mean_test_R2'][best]),
    'cv_train_MAE': float(-search.cv_results_['mean_train_MAE'][best]),
    'train_MAE': train_metrics['MAE'],
    'validation_MAE': val_metrics['MAE'],
    'validation_RMSE': val_metrics['RMSE'],
    'validation_R2': val_metrics['R2'],
    'baseline_validation_MAE': baseline_metrics['MAE'],
    'train_validation_MAE_gap': val_metrics['MAE'] - train_metrics['MAE'],
    'tuning_seconds': train_seconds,
}
display(pd.DataFrame([summary]))
print('Large positive gap suggests overfitting; high train and validation MAE suggests underfitting.')
print('Compare validation MAE with the baseline and other models before choosing a winner.')


## 6 — Save CV results, fitted model, and validation predictions

In [ ]:
if (OUTPUT_DIR / 'final_test_metrics.csv').exists():
    raise FileExistsError('Final test has already been evaluated; preserve that result.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(search.cv_results_).to_csv(OUTPUT_DIR / 'cv_results.csv', index=False)
pd.DataFrame([summary]).to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
(OUTPUT_DIR / 'best_params.json').write_text(json.dumps(search.best_params_, indent=2))
predictions = validation[['cell_id', 'reference_year', TARGET]].copy()
predictions['prediction'] = val_pred
predictions['residual'] = predictions[TARGET] - predictions['prediction']
predictions.to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False)
if 'fixed_risk_class' in validation:
    predictions['fixed_risk_class'] = validation['fixed_risk_class'].to_numpy()
    predictions['absolute_error'] = predictions['residual'].abs()
    risk_errors = predictions.groupby('fixed_risk_class', dropna=False).agg(
        n=(TARGET, 'size'), MAE=('absolute_error', 'mean'),
        mean_residual=('residual', 'mean'))
    risk_errors.to_csv(OUTPUT_DIR / 'validation_by_risk_class.csv')
    display(risk_errors)
joblib.dump(model, OUTPUT_DIR / 'model.joblib')
print('Saved results to', OUTPUT_DIR)


## 7 — Inspect errors and candidate rankings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(validation[TARGET], val_pred, s=6, alpha=0.25)
lo = min(validation[TARGET].min(), val_pred.min())
hi = max(validation[TARGET].max(), val_pred.max())
axes[0].plot([lo, hi], [lo, hi], 'k--')
axes[0].set(xlabel='Observed HRI', ylabel='Predicted HRI', title='Validation: observed vs predicted')
axes[1].scatter(val_pred, validation[TARGET] - val_pred, s=6, alpha=0.25)
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set(xlabel='Predicted HRI', ylabel='Residual', title='Validation residuals')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'validation_diagnostics.png', dpi=160)
plt.show()

# Inspect the search ranking and fold variation; training results are diagnostic only.
columns = ['rank_test_MAE', 'mean_test_MAE', 'std_test_MAE',
           'mean_train_MAE', 'mean_test_RMSE', 'mean_test_R2', 'params']
display(pd.DataFrame(search.cv_results_)[columns].sort_values('rank_test_MAE').head(10))


## 8 — Final test (locked)

Run once on only the selected model after comparing all ten validation results. Do not revise the model after viewing the test result.

In [ ]:
# Run this cell once, only after selecting ONE model using cross-validation and validation.
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    print('Final test is locked. Set RUN_FINAL_TEST=True only for your selected model.')
else:
    if (OUTPUT_DIR / 'final_test_metrics.csv').exists():
        raise FileExistsError('Final test was already evaluated for this model.')
    test = load_split('test')
    if set(test.cell_id) & (set(train.cell_id) | set(validation.cell_id)):
        raise ValueError('A cell_id occurs in multiple frozen splits')
    test_pred = model.predict(test[FEATURES])
    test_metrics = {'model': 'catboost', 'split': 'test', 'n': len(test),
                    **score(test[TARGET], test_pred)}
    pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / 'final_test_metrics.csv', index=False)
    test_predictions = test[['cell_id', 'reference_year', TARGET]].copy()
    test_predictions['prediction'] = test_pred
    test_predictions.to_csv(OUTPUT_DIR / 'final_test_predictions.csv', index=False)
    display(pd.DataFrame([test_metrics]))
